In [1]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset

/home/mila/c/caomeng/miniconda3/envs/trl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
reward_model_path = "/home/mila/c/caomeng/scratch/trl/Qwen/Qwen/Qwen/Qwen3-0.6B-Reward-ultrafeedback"

model = AutoModelForSequenceClassification.from_pretrained(
    reward_model_path,
    num_labels=1,
).to("cuda").eval()

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B",
    padding_side="left"
)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:00<00:00, 958.31it/s, Materializing param=score.weight]


In [7]:
from datasets import load_dataset

ds = load_dataset("trl-lib/ultrafeedback_binarized", split="test")

In [8]:
def build_input(prompt, response):
    text = prompt + "\n" + response
    return text


@torch.no_grad()
def get_reward(texts, batch_size=8):
    all_scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        toks = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=2048,
            return_tensors="pt"
        ).to("cuda")

        out = model(**toks)
        scores = out.logits.squeeze(-1)  # (B,)
        all_scores.append(scores.cpu())

    return torch.cat(all_scores)

In [18]:
chosen_texts = [
    build_input(ex['chosen'][0]["content"], ex['chosen'][1]["content"])
    for ex in ds
]

rejected_texts = [
    build_input(ex['rejected'][0]["content"], ex['rejected'][1]["content"])
    for ex in ds
]

In [19]:
chosen_scores = get_reward(chosen_texts)
rejected_scores = get_reward(rejected_texts)

all_scores = torch.cat([chosen_scores, rejected_scores])

In [23]:
chosen_scores.mean()

tensor(0.9766, dtype=torch.bfloat16)

In [24]:
rejected_scores.mean()

tensor(0.6250, dtype=torch.bfloat16)

In [20]:
mean = all_scores.mean()
std = all_scores.std() + 1e-8

In [21]:
mean

tensor(0.8008, dtype=torch.bfloat16)

In [22]:
std

tensor(0.9727, dtype=torch.bfloat16)

In [30]:
def normalize_to_minus1_1(scores, lo, hi, eps=1e-8):
    # map [lo, hi] -> [-1, 1]
    x = (scores - lo) / (hi - lo + eps)          # -> [0, 1] (roughly)
    x = 2 * x - 1                                 # -> [-1, 1]
    return torch.clamp(x, -1.0, 1.0)

# Robust anchors (change 1/99 to 0.5/99.5 if you want slightly less clipping)

all_scores = all_scores.float()

lo = torch.quantile(all_scores, 0.01)
hi = torch.quantile(all_scores, 0.99)

chosen_norm = normalize_to_minus1_1(chosen_scores, lo, hi)
rejected_norm = normalize_to_minus1_1(rejected_scores, lo, hi)

print("anchors:", float(lo), float(hi))
print("chosen_norm range:", float(chosen_norm.min()), float(chosen_norm.max()))
print("rejected_norm range:", float(rejected_norm.min()), float(rejected_norm.max()))

anchors: -1.5313280820846558 2.984375
chosen_norm range: -1.0 1.0
rejected_norm range: -1.0 1.0


In [34]:
margin = chosen_scores - rejected_scores
margin_all = margin  # (N,)
margin_all = margin_all.float()

m_lo = torch.quantile(margin_all, 0.01)
m_hi = torch.quantile(margin_all, 0.99)

print(m_lo, m_hi)

margin_norm = normalize_to_minus1_1(margin, m_lo, m_hi)
print("margin_norm range:", float(margin_norm.min()), float(margin_norm.max()))

tensor(-1.5159) tensor(2.7814)
margin_norm range: -1.0 1.0


In [31]:
def normalize_to_minus1_1(scores, lo=-1.53, hi=2.98, eps=1e-8):
    scores = scores.float()
    x = (scores - lo) / (hi - lo + eps)
    x = 2 * x - 1
    return torch.clamp(x, -1.0, 1.0)